# Data Crawling — Vietnamese Fake News Detection

Automated, resumable crawling of Vietnamese news articles from the ViFactCheck dataset.

**Sources:** ViFactCheck HuggingFace dataset (`tranthaihoa/vifactcheck`)  
**Output:** Structured JSON files per split (train/dev/test) under `OUTPUT_DIR`  
**Resume:** Re-run this notebook at any time — already-crawled URLs are skipped automatically.

In [1]:
# ============================================================
# CONFIGURATION — edit this cell only
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

CONFIG = {
    # Data source
    "dataset_name": "tranthaihoa/vifactcheck",
    "url_column": "Url",
    "splits": ["train", "dev", "test"],
    "url_limit": None,           # None = all URLs; set int to limit (e.g. 100 for testing)

    # Output
    "output_dir": PROJECT_ROOT / "data" / "json",
    "cache_dir": PROJECT_ROOT / "data" / "caches",
    "output_format": "ml_training",  # options: custom, ml_training, research, detailed

    # Crawl settings
    "max_concurrent": 5,        # parallel requests per split
    "save_interval": 50,         # checkpoint every N completed URLs
    "retry_failed": False,       # set True to retry previously failed URLs
}

In [2]:
import subprocess, sys

_PACKAGES = ["loguru", "tqdm", "beautifulsoup4", "lxml", "httpx",
             "nest-asyncio", "datasets", "Pillow"]

for pkg in _PACKAGES:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"Installed {pkg}")
print("Dependencies ready.")

Installed beautifulsoup4


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installed Pillow
Dependencies ready.


In [3]:
import sys, os
import nest_asyncio
nest_asyncio.apply()

# Add project root to path
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Create output directories
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

from src.crawler.crawler_factory import CrawlerFactory
from src.processing.dataset_handler import DatasetHandler

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {CONFIG['output_dir']}")
print(f"Cache dir:    {CONFIG['cache_dir']}")

Project root: /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection
Output dir:   /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection/data/json
Cache dir:    /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection/data/caches


## Step 1: Load URLs from ViFactCheck Dataset

In [4]:
dataset_handler = DatasetHandler(CONFIG["dataset_name"])

split_urls = {}
for split in CONFIG["splits"]:
    urls = dataset_handler.get_urls_from_split(
        split=split,
        url_column=CONFIG["url_column"],
        limit=CONFIG["url_limit"],
    )
    split_urls[split] = urls
    print(f"  {split:6s}: {len(urls):,} URLs")

total = sum(len(v) for v in split_urls.values())
print(f"\nTotal URLs to process: {total:,}")

--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 5062 URLs to crawl. ---
  train : 5,062 URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 723 URLs to crawl. ---
  dev   : 723 URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 1447 URLs to crawl. ---
  test  : 1,447 URLs

Total URLs to process: 7,232


## Step 2: Check Resume State

In [5]:
import json

for split in CONFIG["splits"]:
    cache_file = CONFIG["cache_dir"] / f"crawling_status_{split}.json"
    failed_file = CONFIG["cache_dir"] / f"failed_urls_{split}.json"

    completed = 0
    failed = 0

    if cache_file.exists():
        with open(cache_file) as f:
            data = json.load(f)
            completed = data.get("length", len(data.get("urls", [])))

    if failed_file.exists():
        with open(failed_file) as f:
            failed = len(json.load(f))

    total = len(split_urls[split])
    remaining = total - completed
    print(f"  {split:6s}: {completed:,} done / {failed:,} failed / {remaining:,} remaining")

print("\nRe-run this notebook to resume. Already-crawled URLs are skipped automatically.")

  train : 4,050 done / 0 failed / 1,012 remaining
  dev   : 0 done / 0 failed / 723 remaining
  test  : 0 done / 0 failed / 1,447 remaining

Re-run this notebook to resume. Already-crawled URLs are skipped automatically.


## Step 3: Crawl Articles

Progress is shown per split. Checkpoints are saved every `save_interval` URLs.
Re-run this cell at any time to resume from where crawling stopped.

In [6]:
async def crawl_split(split: str, urls: list) -> dict:
    """Crawl one split, return summary dict."""
    cache_file = str(CONFIG["cache_dir"] / f"crawling_status_{split}.json")
    failed_file = str(CONFIG["cache_dir"] / f"failed_urls_{split}.json")
    output_file = f"news_data_vifactcheck_{split}.json"

    print(f"\n{'='*50}")
    print(f"Crawling split: {split} ({len(urls):,} URLs)")
    print(f"{'='*50}")

    factory = CrawlerFactory(
        cache_filename=cache_file,
        failed_log_filename=failed_file,
    )

    await factory.crawl_and_save_all(
        urls=urls,
        output_filename=output_file,
        format_name=CONFIG["output_format"],
        max_concurrent=CONFIG["max_concurrent"],
        retry_failed=CONFIG["retry_failed"],
        save_interval=CONFIG["save_interval"],
        output_dir=str(CONFIG["output_dir"]),
    )

    # Return summary
    completed = 0
    if Path(cache_file).exists():
        with open(cache_file) as f:
            data = json.load(f)
            completed = data.get("length", 0)

    failed = 0
    if Path(failed_file).exists():
        with open(failed_file) as f:
            failed = len(json.load(f))

    return {"split": split, "total": len(urls), "completed": completed, "failed": failed}

In [ ]:
import asyncio

summaries = []
for split in CONFIG["splits"]:
    summary = await crawl_split(split, split_urls[split])
    summaries.append(summary)

print("\nAll splits processed.")


Crawling split: train (5,062 URLs)
2026-06-06 14:14:38 | INFO     | src.crawler.crawler_factory:load:55 - Loaded 4050 completed URLs from cache (will skip).
2026-06-06 14:14:38 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:171 - Skipping 0 previously failed URLs (use retry_failed=True to retry).
2026-06-06 14:14:38 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:176 - URLs to crawl: 61 (skipped: 5001, concurrency=5)


Crawling:   7%|▋         | 4/61 [00:30<28:37, 30.13s/it]

2026-06-06 14:15:08 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/giao-duc-huong-nghiep/thi-vao-lop-10-o-ha-noi-con-hoc-den-tram-cam-phu-huynh-lo-phuong-an-du-bi-20230327155051149.htm'): 
2026-06-06 14:15:08 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/phap-luat/thanh-nien-dung-xa-beng-danh-truong-cong-an-xa-bi-thuong-20230330110731678.htm'): 
2026-06-06 14:15:08 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/the-thao/hlv-iraq-noi-mot-dieu-ve-suc-manh-cua-u23-viet-nam-20230320173114638.htm'): 
2026-06-06 14:15:08 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/van-hoa/nhung-buc-tuong-vang-tai-pho-co-hoi-an-bi-boi-ban-20230320142648654.htm'): 
2026-06-06 14:15:08 | ERROR    | helpers.httpx_client:_request:108 - An error occurred whil

Crawling:  15%|█▍        | 9/61 [01:00<07:46,  8.97s/it]

2026-06-06 14:15:38 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/giao-duc-huong-nghiep/thi-vao-lop-10-o-ha-noi-con-hoc-den-tram-cam-phu-huynh-lo-phuong-an-du-bi-20230327155051149.htm'): 
2026-06-06 14:15:38 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/phap-luat/thanh-nien-dung-xa-beng-danh-truong-cong-an-xa-bi-thuong-20230330110731678.htm'): 
2026-06-06 14:15:38 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/xa-hoi/bi-thu-ha-noi-doi-moi-de-nang-cao-chat-luong-phuc-vu-nguoi-dan-20230328200513827.htm'): 
2026-06-06 14:15:38 | ERROR    | helpers.httpx_client:_request:108 - An error occurred while requesting URL('https://dantri.com.vn/suc-khoe/nam-benh-nhan-mang-chum-nho-khong-lo-20-nam-tren-mat-20230309155833682.htm'): 
2026-06-06 14:15:38 | ERROR    | helpers.httpx_client:_request:108 - An error

## Step 4: Results Summary

In [ ]:
print(f"\n{'Split':<8} {'Total':>8} {'Completed':>10} {'Failed':>8} {'Rate':>8}")
print("-" * 50)

for s in summaries:
    rate = f"{s['completed'] / s['total'] * 100:.1f}%" if s['total'] else "—"
    print(f"{s['split']:<8} {s['total']:>8,} {s['completed']:>10,} {s['failed']:>8,} {rate:>8}")

print("-" * 50)
total_urls = sum(s["total"] for s in summaries)
total_done = sum(s["completed"] for s in summaries)
total_fail = sum(s["failed"] for s in summaries)
overall_rate = f"{total_done / total_urls * 100:.1f}%" if total_urls else "—"
print(f"{'TOTAL':<8} {total_urls:>8,} {total_done:>10,} {total_fail:>8,} {overall_rate:>8}")

In [ ]:
print("\nOutput files:")
for split in CONFIG["splits"]:
    out_file = CONFIG["output_dir"] / f"news_data_vifactcheck_{split}.json"
    if out_file.exists():
        with open(out_file) as f:
            count = len(json.load(f))
        size_mb = out_file.stat().st_size / 1024 / 1024
        print(f"  {split:6s}: {out_file.name} — {count:,} articles ({size_mb:.1f} MB)")
    else:
        print(f"  {split:6s}: not yet created")

print("\nNext step: Run notebooks/02_preprocessing.ipynb")